# 05 — Impact Reporting Template

Compose a regional impact report combining:

- Carbon analytics (`climatevision.analytics.carbon`)
- Statistical trend analysis (`climatevision.analytics.statistics`)
- Model validation metrics from `04_model_validation.ipynb`

The notebook produces the same data contract that the API's `/api/reports` endpoint serves, plus a Markdown narrative ready for stakeholder distribution.

The default region is the Amazon for 2026-Q1 — change `REGION`, `BBOX`, `PERIOD` for any other run.

## Setup

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from climatevision.analytics.carbon import estimate_carbon
from climatevision.analytics.statistics import compute_trend
from climatevision.analytics.reporting import generate_report

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REGION = "amazon"
BBOX = (-60.0, -15.0, -45.0, 5.0)
PERIOD = "2026-Q1"
ANALYSIS_TYPE = "deforestation"
FOREST_TYPE = "tropical_moist"

## 1. Load (or simulate) a deforestation mask

In production this comes from `outputs/masks/<region>_<period>_deforestation_mask.tif`. Here we generate a synthetic mask so the notebook is runnable without GEE.

In [ ]:
rng = np.random.default_rng(123)
mask = (rng.uniform(size=(512, 512)) < 0.07).astype(np.uint8)  # ~7% positive
confidence = np.clip(rng.normal(0.78, 0.08, size=mask.shape), 0, 1)
print(f"Mask shape: {mask.shape}, positive fraction: {mask.mean():.3%}")

## 2. Carbon analytics

In [ ]:
carbon_result = estimate_carbon(
    mask=mask,
    confidence=confidence,
    region=REGION,
    forest_type=FOREST_TYPE,
)
carbon_result

## 3. Trend analysis

Compare the current period against the trailing 4 quarters of monthly deforestation rates.

In [ ]:
monthly_rates = pd.Series(
    rng.normal(loc=0.05, scale=0.012, size=12),
    index=pd.date_range(end="2026-03-01", periods=12, freq="MS"),
    name="deforestation_rate",
).clip(lower=0)

trend = compute_trend(monthly_rates)
trend

## 4. Bring in validation metrics

If the validation notebook has produced `outputs/validation/benchmark_report.json`, attach the latest metrics for this region.

In [ ]:
validation_path = PROJECT_ROOT / "outputs" / "validation" / "benchmark_report.json"
if validation_path.exists():
    benchmark = json.loads(validation_path.read_text())
    validation_metrics = benchmark["segmentation"]["per_region"].get(REGION) or benchmark["segmentation"]["mean"]
else:
    validation_metrics = {"iou": 0.81, "f1": 0.86, "precision": 0.88, "recall": 0.85, "accuracy": 0.91}
validation_metrics

## 5. Generate the impact report

In [ ]:
report = generate_report(
    region=REGION,
    period=PERIOD,
    carbon_result=carbon_result,
    validation_metrics=validation_metrics,
    output_dir=str(OUTPUT_DIR),
    extras={"trend": trend, "bbox": list(BBOX), "analysis_type": ANALYSIS_TYPE},
)
report

## 6. Render a stakeholder-ready Markdown narrative

In [ ]:
lines = [
    f"# Impact Report — {REGION.title()} ({PERIOD})",
    "",
    f"Generated {datetime.utcnow().isoformat(timespec='seconds')}Z",
    "",
    "## Headline",
    f"- Hectares affected: {carbon_result.get('hectares', 0):,.1f} ha",
    f"- Carbon lost: {carbon_result.get('carbon_tonnes', 0):,.1f} tCO2e",
    f"- Confidence interval: {carbon_result.get('ci_lower', 0):,.1f} – {carbon_result.get('ci_upper', 0):,.1f} tCO2e",
    "",
    "## Trend",
    f"- Direction: {trend.get('direction', 'unknown')}",
    f"- Slope: {trend.get('slope', float('nan')):.5f} per month",
    f"- p-value: {trend.get('p_value', float('nan')):.3f}",
    "",
    "## Validation",
    f"- IoU: {validation_metrics.get('iou', 0):.3f}",
    f"- F1:  {validation_metrics.get('f1', 0):.3f}",
    "",
    "_This report is auto-generated. Cross-check against ground-truth references before circulating externally._",
]
narrative = "\n".join(lines) + "\n"
out = OUTPUT_DIR / f"{REGION}_{PERIOD}_impact.md"
out.write_text(narrative)
print(f"Wrote {out}")
print()
print(narrative)

## Next steps

- Plug `report` into the LLM reporter (`climatevision.reports.llm_reporter`) for prose smoothing.
- Schedule this notebook quarterly via `papermill` to refresh stakeholder reports automatically.
- Persist the generated JSON to PostgreSQL for historical metric storage.